In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../player_stats.csv')

In [ ]:
df.head(10)

In [ ]:
cols_to_drop = [
    'description', 
    'image', 
    'international_reputation', 
    'body_type',
    'real_face',
    'specialities',
    'club_id',
    'club_league_id',
    'club_logo',
    'club_rating',
    'club_kit_number',
    'club_joined',
    'country_id',
    'country_league_id',
    'country_league_name',
    'country_flag',
    'country_rating',
    'country_position',
    'country_kit_number',
    'play_styles',
    'url'
    ]

df.drop(columns=cols_to_drop, inplace=True)
df.head(10)

In [ ]:
#Drop rows with missing country_name
df.dropna(subset=['country_name'], inplace=True)

#Convert all numeric columns to doubles
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[numeric_cols] = df[numeric_cols].astype('float64')

#Convert defending_defensive_awareness, defending_standing_tackle to numeric doubles safely
df['defending_defensive_awareness'] = pd.to_numeric(df['defending_defensive_awareness'], errors='coerce').astype('float64')
df['defending_standing_tackle'] = pd.to_numeric(df['defending_standing_tackle'], errors='coerce').astype('float64')

#convert dob and club_contract_valid_until to datetime
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['club_contract_valid_until'] = pd.to_datetime(df['club_contract_valid_until'], errors='coerce')

#Change preferred_foot to binary
df['preferred_foot'] = df['preferred_foot'].map({'Left': 0, 'Right': 1})

df.head(10)

In [ ]:
for col in df.columns:
    print(col, df[col].astype(str).str.contains("sofifa.com").mean())

In [ ]:
import numpy as np
import pandas as pd

cols_to_drop_final = [
    "defending_defensive_awareness",
    "defending_standing_tackle",
    "mentality_attack_position",
]

# Drop the columns (if present)
df = df.drop(columns=cols_to_drop_final, errors="ignore")

# If release_clause is missing, set it to 0
if "release_clause" in df.columns:
    df["release_clause"] = pd.to_numeric(df["release_clause"], errors="coerce").fillna(0)

# Fill missing club_name and club_league_name with "Free Agent"
if "club_name" in df.columns:
    df["club_name"] = df["club_name"].fillna("Free Agent")

if "club_league_name" in df.columns:
    df["club_league_name"] = df["club_league_name"].fillna("Free Agent")

# Fill missing club_position with "FREE"
if "club_position" in df.columns:
    df["club_position"] = df["club_position"].fillna("FREE")

# Fill missing club_contract_valid_until with 1900-01-01 to indicate no contract
if "club_contract_valid_until" in df.columns:
    # Convert to datetime first, then fill missing with sentinel date
    df["club_contract_valid_until"] = pd.to_datetime(df["club_contract_valid_until"], errors="coerce")
    df["club_contract_valid_until"] = df["club_contract_valid_until"].fillna(pd.Timestamp("1900-01-01"))

    # (Recommended) explicit flag so the model can learn “no contract” directly
    df["no_contract"] = (df["club_contract_valid_until"] == pd.Timestamp("1900-01-01")).astype(int)

# GK columns: if positions does not contain "GK", set GK stats to 0
gk_cols = [
    "goalkeeping_gk_diving",
    "goalkeeping_gk_handling",
    "goalkeeping_gk_kicking",
    "goalkeeping_gk_positioning",
    "goalkeeping_gk_reflexes",
]

if "positions" in df.columns:
    is_gk = df["positions"].astype(str).str.contains("GK", na=False)

    for col in gk_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = np.where(is_gk, df[col], 0)

# Create binary free agent column
if "club_name" in df.columns and "club_league_name" in df.columns:
    df["free_agent"] = np.where(
        (df["club_name"] == "Free Agent") & (df["club_league_name"] == "Free Agent"),
        1,
        0,
    )

In [ ]:
df.head(100)

In [ ]:
df.to_csv('../player_stats_cleaned.csv', index=False)